# Publication copy
The code and analytical commentary are preserved. Stored execution outputs were cleared to avoid publishing respondent previews or local execution details. Vetted aggregate tables and figures are provided in `course_artifacts`; see the repository results guide. The fully executed originals remain in the local course workspace. This notebook needs the separately acquired survey data and protected fit caches for a complete replay. No models were refitted for publication.


# Complete responses and a reduced overall-worry feature

Version 3 uses only respondents with all six worry and all seven harm questions answered. No imputation is used. We retain six within-person centred worry features and add the respondent's overall worry on the same recoded, range-normalized scale. Personal and known-person harm remain separate binary indicators.

**Scientific rationale:** centring describes which domains dominate but removes overall worry. The additional mean distinguishes uniformly unworried from uniformly worried respondents. Its multiplier equals that of each centred worry feature: its squared-distance contribution is one-sixth of the contribution needed to reproduce uncentred worry distances. This is an explicit exploratory weighting choice, not an estimated psychological scale.

We retain 0–1–2 as baseline, 0–1–3 as the principal alternative and 0–1–5 as sensitivity; three worry/harm balances; k=2–10; and 50 respondent plus 50 stratified-country repetitions per candidate. All 8,100 repetition fits are new. Held-out countries remain untouched. Version 2 is preserved. Differences from v2 combine cohort and representation changes and cannot identify their separate effects.

## 1. Fix the design before examining results

**What and why:** declare features, encodings, seeds, balances and repetitions. Equal-country weights preserve relative survey weights within each retained country. Recalculate them on each training subset. The nominal block multipliers are retained for comparison with v2; adding the mean increases actual worry distance rather than silently reducing the centred-feature multiplier. ARI is unweighted, unlike clustering and prevalence summaries. Cached fits are accepted only if their design fingerprint matches.

In [ ]:
from pathlib import Path
import json, hashlib, itertools
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
from scipy.optimize import linear_sum_assignment
from joblib import Parallel, delayed, parallel_config
from threadpoolctl import threadpool_limits

ROOT=Path.cwd()
if not (ROOT/'course_artifacts').exists(): ROOT=ROOT.parent
INPUT=ROOT/'course_artifacts/week04_search_v4/stage1_eligible_development.parquet'
OUTPUT=ROOT/'course_artifacts/week04_experiment1_complete_overall_worry_v3'
OUTPUT.mkdir(exist_ok=True)
(OUTPUT/'fits').mkdir(exist_ok=True)
WORRY=['worry_'+d for d in ['food','water','crime','severe_weather','traffic','mental_health']]
DOMAINS=[c.removeprefix('worry_') for c in WORRY]+['work']
PERSONAL=['harm_'+d+'_personal' for d in DOMAINS]
KNOWN=['harm_'+d+'_known_person' for d in DOMAINS]
FEATURES=WORRY+PERSONAL+KNOWN
BALANCES={'balanced':.50,'worry_emphasized':.75,'harm_emphasized':.25}
K_VALUES=range(2,11)
REPETITIONS=50
SEED=20261005
N_JOBS=4
DESIGN={'balances':BALANCES,'k':list(K_VALUES),'repetitions':REPETITIONS,'fraction':.8,'seed':SEED,'n_init':20,'imputation':'none','eligibility':'all_20_derived_features_observed','overall_worry':'mean_with_same_multiplier_as_each_centred_feature','strata':['GlobalRegion','CountryIncomeLevel']}
FINGERPRINT=hashlib.sha256(INPUT.read_bytes()+json.dumps(DESIGN,sort_keys=True).encode()).hexdigest()
(OUTPUT/'design.json').write_text(json.dumps(DESIGN,indent=2))
sns.set_theme(style='whitegrid')
def show(title,table):
    display(Markdown('### '+title)); display(table)
def country_weights(frame):
    return (frame.WGT/frame.groupby('COUNTRY_ISO3').WGT.transform('sum')/frame.COUNTRY_ISO3.nunique()).to_numpy()


## 2. Restrict to complete responses and audit exclusions

**What this cell does:** first identifies the previous eligible cohort, then retains only complete worry and harm responses. It records losses overall and by country, checks identifiers and valid values, and recalculates equal-country weights.

**Why:** the new clustering does not depend on imputed answers. Complete cases may nevertheless be selected systematically; keeping all countries does not remove selection bias within countries. Percentages excluded under old weights describe the original population; newly calculated weights describe the retained one.

In [ ]:
raw=pd.read_parquet(INPUT)
assert raw.country_split.eq('train').all()
eligible=raw[WORRY].notna().sum(axis=1).ge(4)&raw[PERSONAL].notna().sum(axis=1).ge(4)
previous=raw.loc[eligible].copy()
complete=previous[FEATURES].notna().all(axis=1)
previous_weights=country_weights(previous)
data=previous.loc[complete].copy().reset_index(drop=True)
assert np.isfinite(data[FEATURES].to_numpy(float)).all()
assert data[WORRY].isin([0,1,2]).all().all()
assert data[PERSONAL+KNOWN].isin([0,1]).all().all()
audit=pd.DataFrame({'previous_eligible':[len(previous)],'retained':[len(data)],'excluded':[int((~complete).sum())],
                    'excluded_pct':[100*(~complete).mean()],
                    'excluded_previous_weight_pct':[100*previous_weights[~complete.to_numpy()].sum()]})
audit.to_csv(OUTPUT/'complete_case_audit.csv',index=False)
country_audit=previous.assign(complete=complete).groupby('COUNTRY_ISO3').complete.agg(['size','sum','mean'])
country_audit.columns=['previous_eligible','retained','retained_fraction']
country_audit.to_csv(OUTPUT/'complete_case_country_audit.csv')
show('Complete-response retention and exclusions',audit)
show('Country-level complete-response retention',country_audit)
assert data.WPID_RANDOM.is_unique
assert np.isfinite(data.WGT).all() and data.WGT.gt(0).all()
weights=country_weights(data)
countries=data.COUNTRY_ISO3.to_numpy()
country_rows={c:np.flatnonzero(countries==c) for c in sorted(set(countries))}
registry=pd.read_csv(ROOT/'course_artifacts/week03_country_split.csv')
registry=registry.loc[registry.country_split.eq('train')&registry.COUNTRY_ISO3.isin(country_rows)].copy()
assert len(registry)==len(country_rows) and registry.COUNTRY_ISO3.is_unique
show('Fixed development population',pd.DataFrame({'respondents':[len(data)],'countries':[len(country_rows)],'excluded_from_input':[len(raw)-len(data)]}))


## 3. Stratified country quotas and reproducible sampling plans

**What and why:** cross region and income categories, retaining at least one country in each stratum. Income code 9 remains a separate “not classified” category. Start with floor(80% of each stratum), then allocate remaining places by largest remainder with deterministic ties. The total is 90 countries. Small strata cannot all retain exactly 80%; some countries are always included. This limits what we can learn about their omission. This crossed-stratum perturbation is distinct from the original marginally balanced development/held-out split.

For respondent sampling, select approximately 80% without replacement within every country. All candidate models use exactly the same sampling plans, making comparisons fair.

In [ ]:
groups=registry.groupby(['GlobalRegion','CountryIncomeLevel'],sort=True)
quota=groups.size().rename('available').to_frame()
quota['selected']=np.maximum(1,np.floor(.8*quota.available).astype(int))
target=round(.8*len(registry))
while quota.selected.sum()<target:
    remainder=(.8*quota.available-quota.selected).where(quota.selected<quota.available,-np.inf)
    quota.loc[remainder.idxmax(),'selected']+=1
quota['always_included']=quota.selected.eq(quota.available)
quota.to_csv(OUTPUT/'stratum_quotas.csv')
show('Country retention by region × income stratum',quota)
print('Countries per repetition:',target,'; always included:',int(quota.loc[quota.always_included,'available'].sum()))
plans={'respondents':[],'countries':[]}
plan_rows=[]
for repeat in range(REPETITIONS):
    rng=np.random.default_rng(SEED+repeat)
    rows=np.sort(np.concatenate([rng.choice(idx,round(.8*len(idx)),replace=False) for idx in country_rows.values()]))
    plans['respondents'].append(rows)
    selected=[]
    for key,frame in groups:
        selected.extend(rng.choice(sorted(frame.COUNTRY_ISO3),int(quota.loc[key,'selected']),replace=False))
    rows=np.flatnonzero(np.isin(countries,selected))
    plans['countries'].append(rows)
    for kind in plans:
        idx=plans[kind][-1]
        plan_rows.append({'experiment':kind,'repeat':repeat,'respondents':len(idx),'countries':len(set(countries[idx])),'country_list':' '.join(sorted(set(countries[idx])))})
pd.DataFrame(plan_rows).to_csv(OUTPUT/'sampling_plans.csv',index=False)
joblib.dump(plans,OUTPUT/'sampling_indices.joblib')


## 4. Encode worry, centre its shape, and retain its intensity

**What this cell does:** recodes observed worry, divides by the maximum, subtracts each person's mean from their six values, and appends that mean as a seventh worry feature. No transformation is learned from missing values because none remain. The mean is on the selected encoding, not always the original 0–1–2 scale.

**Why:** shape and intensity are now independently visible. Six centred values plus the mean retain all worry information. Equal multiplier does not reproduce uncentred distance: that would require multiplying the mean by √6. Harm features are uncentred. Analysis weights remain positive and are never centred. Checks confirm the transformation and demonstrate the mathematical distance identity.

In [ ]:
ENCODINGS={'012':2.,'013':3.,'015':5.}
DESIGN.update(encodings=ENCODINGS,overall_worry_multiplier='sqrt(alpha)/sqrt(6)')
FINGERPRINT=hashlib.sha256(INPUT.read_bytes()+json.dumps(DESIGN,sort_keys=True).encode()).hexdigest()
(OUTPUT/'design.json').write_text(json.dumps(DESIGN,indent=2))
def encoded(frame,top):
    values=frame[FEATURES].to_numpy(float).copy()
    values[:,:6]=np.where(values[:,:6]==2,top,values[:,:6])
    return values
def transform(frame,alpha,top):
    values=encoded(frame,top)
    worry=values[:,:6]/top
    overall=worry.mean(axis=1,keepdims=True)
    relative=worry-overall
    return np.column_stack([relative*np.sqrt(alpha)/np.sqrt(6),
                            overall*np.sqrt(alpha)/np.sqrt(6),
                            values[:,6:]*np.sqrt((1-alpha)/2)/np.sqrt(7)])
for top in ENCODINGS.values():
    worry=encoded(data,top)[:,:6]/top
    overall=worry.mean(axis=1,keepdims=True)
    relative=worry-overall
    assert np.allclose(relative.mean(axis=1),0)
    assert np.allclose(relative+overall,worry)
    delta=worry[:100]-worry[100:200]
    shape=relative[:100]-relative[100:200]
    intensity=overall[:100]-overall[100:200]
    assert np.allclose((delta**2).sum(axis=1),(shape**2).sum(axis=1)+6*intensity[:,0]**2)
all_references={}
for encoding,top in ENCODINGS.items():
    for balance,alpha in BALANCES.items():
        matrix=transform(data,alpha,top)
        assert matrix.shape==(len(data),21)
        for k in K_VALUES:
            path=OUTPUT/f'reference_{encoding}_{balance}_k{k}.joblib'
            result=joblib.load(path) if path.exists() else {}
            if result.get('fingerprint')==FINGERPRINT:
                labels=result['labels']
            else:
                with threadpool_limits(limits=1):
                    model=KMeans(n_clusters=k,n_init=20,random_state=20231006).fit(matrix,sample_weight=weights)
                labels=model.labels_.astype('int8')
                joblib.dump({'fingerprint':FINGERPRINT,'labels':labels,'centres':model.cluster_centers_},path)
            all_references[(encoding,balance,k)]=labels
    print('Reference solutions ready:',encoding,flush=True)


## 5. ARI and label-independent cluster recovery

**What and why:** count pairs placed together or apart in two partitions, correcting agreement for chance given cluster sizes. The contingency table contains counts for each pair of labels. We calculate exactly the standard adjusted Rand index using this table, avoiding repeated sorting for thousands of comparisons. Tests below check agreement with scikit-learn. ARI=1 means identical partitions up to label names; near zero means chance-level agreement under this correction, and negative values are possible.

For individual cluster recovery, Jaccard = overlap / union. A one-to-one assignment maximizes total Jaccard between reference and repetition clusters. It may still match a poorly recovered cluster: we report the score rather than treating matching as evidence of recovery.

In [ ]:
def contingency(a,b,k):
    return np.bincount(a.astype('int64')*k+b,minlength=k*k).reshape(k,k)
def ari(a,b,k):
    counts=contingency(a,b,k).astype(float)
    choose=lambda x: x*(x-1)/2
    total=choose(len(a))
    if total==0: return 1.0
    agreement=choose(counts).sum()
    first=choose(counts.sum(axis=1)).sum(); second=choose(counts.sum(axis=0)).sum()
    expected=first*second/total
    denominator=(first+second)/2-expected
    return 1.0 if denominator==0 else (agreement-expected)/denominator
rng=np.random.default_rng(SEED)
for k in K_VALUES:
    for size in [2,100,1000]:
        a=rng.integers(k,size=size); b=rng.integers(k,size=size)
        assert np.isclose(ari(a,b,k),adjusted_rand_score(a,b))
assert ari(np.zeros(10,dtype=int),np.ones(10,dtype=int),2)==1
def match(reference,labels,k):
    counts=contingency(reference,labels,k)
    union=counts.sum(axis=1)[:,None]+counts.sum(axis=0)[None,:]-counts
    scores=np.divide(counts,union,out=np.zeros_like(counts,dtype=float),where=union>0)
    rows,cols=linear_sum_assignment(-scores)
    mapping=np.empty(k,dtype=int); mapping[cols]=rows
    return mapping[labels],scores[rows,cols]
print('ARI implementation verified against scikit-learn.')


## 6. Refit each candidate under both sampling tests

**What and why:** fit 50 respondent subsamples and 50 region × income-stratified country subsamples per candidate. The plans are shared across encodings and balances. Every model predicts the same complete-response cohort, giving comparable partitions even when its training countries differ. Country weights are recalculated within each training sample. There is no imputer or fitted scaler. Twenty initializations reduce optimization sensitivity. Independent caches permit safe resumption.

In [ ]:
def fit_one(encoding,kind,repeat,balance,k):
    path=OUTPUT/'fits'/f'{encoding}_{kind}_{repeat:02d}_{balance}_k{k}.joblib'
    if path.exists():
        result=joblib.load(path)
        if result['fingerprint']==FINGERPRINT: return result['labels']
    training=data.iloc[plans[kind][repeat]]
    top=ENCODINGS[encoding]; alpha=BALANCES[balance]
    with threadpool_limits(limits=1):
        model=KMeans(n_clusters=k,n_init=20,random_state=SEED+1000+repeat)
        model.fit(transform(training,alpha,top),sample_weight=country_weights(training))
        labels=model.predict(transform(data,alpha,top)).astype('int8')
    joblib.dump({'fingerprint':FINGERPRINT,'labels':labels,'iterations':model.n_iter_},path)
    return labels
all_results={}
for encoding in ENCODINGS:
    for kind in plans:
        for balance in BALANCES:
            for k in K_VALUES:
                cached = all((OUTPUT/'fits'/f'{encoding}_{kind}_{r:02d}_{balance}_k{k}.joblib').exists() for r in range(REPETITIONS))
                if cached:
                    labels=[fit_one(encoding,kind,r,balance,k) for r in range(REPETITIONS)]
                else:
                    with parallel_config(backend='loky',inner_max_num_threads=1):
                        labels=Parallel(n_jobs=N_JOBS)(delayed(fit_one)(encoding,kind,r,balance,k) for r in range(REPETITIONS))
                all_results[(encoding,kind,balance,k)]=labels
                print(f'Completed {encoding}: {kind}, {balance}, k={k}',flush=True)


## 7. Measure partition stability and profile consistency

**What and why:** compute 50 reference ARIs and 1,225 dependent pairwise ARIs per candidate and sampling test. Country comparisons additionally use jointly represented countries. One-to-one Jaccard matching tracks individual clusters; descriptions use observed original worry and harm values, with overall worry included explicitly. High ARI measures reproducibility, not scientific usefulness or proof of distinct populations. The lower decile describes the weaker 10% of comparisons, not a confidence bound.

In [ ]:
# Cache country membership once rather than repeatedly comparing text labels.
country_names=sorted(country_rows)
country_lookup={country:i for i,country in enumerate(country_names)}
country_codes=np.array([country_lookup[country] for country in countries])
training_country_sets={kind:[set(countries[idx]) for idx in indices] for kind,indices in plans.items()}
training_country_masks={kind:[np.array([country in selected for country in country_names]) for selected in selections] for kind,selections in training_country_sets.items()}
# Verify the faster bookkeeping selects exactly the same respondents.
for kind in plans:
    assert np.array_equal(training_country_masks[kind][0][country_codes],np.isin(countries,list(training_country_sets[kind][0])))
MAIN_OUTPUT=OUTPUT
for encoding in ENCODINGS:
    OUTPUT=MAIN_OUTPUT/encoding; OUTPUT.mkdir(exist_ok=True)
    references={(b,k):v for (e,b,k),v in all_references.items() if e==encoding}
    results={(kind,b,k):v for (e,kind,b,k),v in all_results.items() if e==encoding}
    reference_rows=[]; pair_rows=[]; recovery_rows=[]; profile_rows=[]
    display_values=data[FEATURES].to_numpy(float)
    fixed_worry=data[WORRY].to_numpy(float)
    relative_worry=fixed_worry-fixed_worry.mean(axis=1,keepdims=True)
    description=np.column_stack([display_values,relative_worry,fixed_worry.mean(axis=1)])
    description_names=FEATURES+['relative_'+c for c in WORRY]+['overall_worry_original_scale']
    def means(labels,k):
        values=np.full((k,len(description_names)),np.nan)
        for j in range(len(description_names)):
            observed=np.isfinite(description[:,j])
            denominator=np.bincount(labels[observed],weights=weights[observed],minlength=k)
            numerator=np.bincount(labels[observed],weights=weights[observed]*description[observed,j],minlength=k)
            values[:,j]=np.divide(numerator,denominator,out=np.full(k,np.nan),where=denominator>0)
        return values
    for (kind,balance,k),labels_list in results.items():
        reference=references[(balance,k)]
        base_means=means(reference,k)
        for repeat,labels in enumerate(labels_list):
            shared=training_country_masks[kind][repeat][country_codes]
            aligned,jaccard=match(reference,labels,k)
            reference_rows.append({'experiment':kind,'balance':balance,'k':k,'repeat':repeat,'ari':ari(reference,labels,k),'shared_country_ari':ari(reference[shared],labels[shared],k) if kind=='countries' else np.nan})
            current_means=means(aligned,k)
            prevalence=np.bincount(aligned,weights=weights,minlength=k)
            for cluster in range(k):
                recovery_rows.append({'experiment':kind,'balance':balance,'k':k,'repeat':repeat,'cluster':cluster,'jaccard':jaccard[cluster],'prevalence':prevalence[cluster]})
                for j,feature in enumerate(description_names):
                    profile_rows.append({'experiment':kind,'balance':balance,'k':k,'repeat':repeat,'cluster':cluster,'feature':feature,'mean':current_means[cluster,j],'reference_mean':base_means[cluster,j]})
        for first,second in itertools.combinations(range(REPETITIONS),2):
            a,b=labels_list[first],labels_list[second]
            row={'experiment':kind,'balance':balance,'k':k,'first':first,'second':second,'ari':ari(a,b,k)}
            if kind=='countries':
                common=training_country_sets[kind][first]&training_country_sets[kind][second]
                shared=(training_country_masks[kind][first]&training_country_masks[kind][second])[country_codes]
                row.update(shared_country_ari=ari(a[shared],b[shared],k),shared_countries=len(common))
            pair_rows.append(row)
        print('Compared',kind,balance,k,flush=True)
    reference_table=pd.DataFrame(reference_rows); pair_table=pd.DataFrame(pair_rows)
    recovery_table=pd.DataFrame(recovery_rows); profile_table=pd.DataFrame(profile_rows)
    for name,table in [('reference_ari',reference_table),('pairwise_ari',pair_table),('cluster_recovery',recovery_table),('profile_repetitions',profile_table)]:
        table.to_csv(OUTPUT/f'{name}.csv',index=False)
    def summarize(table,value='ari'):
        return table.groupby(['experiment','balance','k'])[value].agg(median='median',p10=lambda x:x.quantile(.1),minimum='min',maximum='max').reset_index()
    summary=summarize(reference_table).merge(summarize(pair_table),on=['experiment','balance','k'],suffixes=('_reference','_pairwise'))
    recovery_summary=recovery_table.groupby(['experiment','balance','k']).agg(median_cluster_jaccard=('jaccard','median'),p10_cluster_jaccard=('jaccard',lambda x:x.quantile(.1)))
    summary=summary.merge(recovery_summary,on=['experiment','balance','k'])
    summary.to_csv(OUTPUT/'stability_summary.csv',index=False)
    show('Reference and pairwise ARI distributions: all 54 candidate–perturbation combinations',summary)
    show('Country comparisons restricted to jointly represented countries',summarize(pair_table.loc[pair_table.experiment.eq('countries')],'shared_country_ari'))
    fig,axes=plt.subplots(2,3,figsize=(15,8),sharex=True,sharey=True)
    for row,kind in enumerate(plans):
        for col,balance in enumerate(BALANCES):
            subset=summary.loc[summary.experiment.eq(kind)&summary.balance.eq(balance)]
            ax=axes[row,col]
            ax.plot(subset.k,subset.median_pairwise,'o-',label='Pairwise median')
            ax.plot(subset.k,subset.p10_pairwise,'o--',label='Pairwise lower decile')
            ax.plot(subset.k,subset.median_reference,':',label='Reference median')
            ax.set(title=f'{kind}: {balance}',xlabel='Number of clusters (k)',ylabel='Adjusted Rand index',xticks=list(K_VALUES))
    axes[0,0].legend(fontsize=8)
    fig.tight_layout(); fig.savefig(OUTPUT/'ari_distributions.png',dpi=160); plt.show()
    display(Markdown('**Interpretation:** examine both perturbations and the lower tail before interpreting a high median. No horizontal pass threshold is imposed.'))
OUTPUT=MAIN_OUTPUT

## 8. Characterize all reference clusters, not just stable ones

**What and why:** report observed worry-category percentages, personal and known-person harm, overall-worry distributions, and relative-worry dispersion for every candidate. There are no imputed values. A flat average may conceal opposing individual patterns, so both distributions and averages matter. These descriptions must support any eventual profile names.

In [ ]:
category_rows=[]; feature_rows=[]; burden_rows=[]; description_rows=[]
observed=data[FEATURES].to_numpy(float)
completed=data[FEATURES].to_numpy(float)
relative=completed[:,:6]-completed[:,:6].mean(axis=1,keepdims=True)
burdens={'overall_worry_original_scale':completed[:,:6].mean(axis=1),'personal_harm_domains_observed':completed[:,6:13].sum(axis=1),'known_person_harm_domains_observed':completed[:,13:].sum(axis=1)}
def weighted_quantile(values,w,q):
    order=np.argsort(values); return np.interp(q,np.cumsum(w[order])/w.sum(),values[order])
for (encoding,balance,k),labels in all_references.items():
    for cluster in range(k):
        inside=labels==cluster; outside=~inside
        key={'encoding':encoding,'balance':balance,'k':k,'cluster':cluster}
        share=weights[inside].sum()
        for j,feature in enumerate(FEATURES):
            valid=np.isfinite(observed[:,j]); current=inside&valid; rest=outside&valid
            mean=np.average(observed[current,j],weights=weights[current])
            other=np.average(observed[rest,j],weights=weights[rest])
            feature_rows.append({**key,'feature':feature,'mean':mean,'rest_mean':other,'contrast':mean-other,'missing_share':weights[inside&~valid].sum()/share})
            if feature in WORRY:
                for value in [0,1,2]:
                    category_rows.append({**key,'feature':feature,'response':value,'proportion_observed':weights[current&(observed[:,j]==value)].sum()/weights[current].sum(),'missing_share':weights[inside&~valid].sum()/share})
        for feature,values in burdens.items():
            q=weighted_quantile(values[inside],weights[inside],np.array([.1,.5,.9]))
            burden_rows.append({**key,'measure':feature,'mean':np.average(values[inside],weights=weights[inside]),'p10':q[0],'median':q[1],'p90':q[2]})
        rel=np.average(relative[inside],axis=0,weights=weights[inside])
        dispersion=np.average((relative[inside]-rel)**2,axis=0,weights=weights[inside])**.5
        harm=np.average(completed[inside,6:],axis=0,weights=weights[inside])
        focus=np.argsort(rel)[-2:][::-1]
        strongest=int(np.argmax(harm))
        description_rows.append({**key,'equal_country_share':share,'top_relative_worry':', '.join(WORRY[i].removeprefix('worry_') for i in focus),'highest_relative_worry':rel[focus[0]],'lowest_relative_worry':rel.min(),'mean_within_cluster_relative_worry_sd':dispersion.mean(),'highest_harm_feature':(PERSONAL+KNOWN)[strongest],'highest_harm_prevalence_observed':harm[strongest],'description':f"Relative worry emphasis: {WORRY[focus[0]]} ({rel[focus[0]]:+.2f}) and {WORRY[focus[1]]} ({rel[focus[1]]:+.2f}); strongest harm indicator: {(PERSONAL+KNOWN)[strongest]} ({harm[strongest]:.0%})."})
categories=pd.DataFrame(category_rows); features=pd.DataFrame(feature_rows)
burden=pd.DataFrame(burden_rows); descriptions=pd.DataFrame(description_rows)
for name,table in [('worry_categories_all_candidates',categories),('feature_contrasts_all_candidates',features),('burden_distributions_all_candidates',burden),('profile_descriptions_all_candidates',descriptions)]:
    table.to_csv(OUTPUT/f'{name}.csv',index=False)
show('Descriptions for all balanced solutions across encodings',descriptions.loc[descriptions.balance.eq('balanced')])


## 9. Visual profiles for every candidate and paired encoding comparison

**What and why:** create one readable figure for each of the 81 solutions, showing percentages very worried and each harm indicator. Worry category distributions and intensity distributions are available in the saved tables. These original-unit plots are comparable across encodings. The encoding comparison shows how much changing spacing changes assignments and whether stability improves or worsens. Neither a higher ARI nor a more attractive story automatically selects a winner.

The overall-worry distributions are especially important: the added feature should separate intensity without erasing thematic differences. Comparisons against v2 are exploratory and confound complete-case selection with the added feature.

In [ ]:
PLOTS=OUTPUT/'profiles'; PLOTS.mkdir(exist_ok=True)
for encoding,balance,k in all_references:
    subset=features.loc[features.encoding.eq(encoding)&features.balance.eq(balance)&features.k.eq(k)]
    worry=categories.loc[categories.encoding.eq(encoding)&categories.balance.eq(balance)&categories.k.eq(k)&categories.response.eq(2)]
    fig,axes=plt.subplots(1,3,figsize=(16,5))
    for ax,columns,title in zip(axes,[WORRY,PERSONAL,KNOWN],['Very worried among observed answers','Personal harm among observed answers','Known-person harm among observed answers']):
        table=(worry.pivot(index='cluster',columns='feature',values='proportion_observed') if columns==WORRY else subset.pivot(index='cluster',columns='feature',values='mean'))
        sns.heatmap(table.reindex(columns=columns)*100,annot=True,fmt='.0f',vmin=0,vmax=100,cmap='Blues',ax=ax,cbar=False)
        ax.set(title=title,xlabel='Risk domain',ylabel='Reference cluster')
        ax.set_xticklabels([d.replace('_',' ') for d in (DOMAINS[:6] if columns==WORRY else DOMAINS)],rotation=40,ha='right')
    fig.suptitle(f'Encoding {encoding} | {balance} | k={k} — percentages')
    fig.tight_layout(); fig.savefig(PLOTS/f'{encoding}_{balance}_k{k}.png',dpi=140); plt.close(fig)
summaries=[]; comparisons=[]
for encoding in ENCODINGS:
    table=pd.read_csv(OUTPUT/encoding/'stability_summary.csv'); table.insert(0,'encoding',encoding); summaries.append(table)
    for balance in BALANCES:
        for k in K_VALUES:
            comparisons.append({'encoding':encoding,'balance':balance,'k':k,'ari_against_012':ari(all_references[(encoding,balance,k)],all_references[('012',balance,k)],k)})
summary_all=pd.concat(summaries,ignore_index=True)
summary_all.to_csv(OUTPUT/'stability_all_encodings.csv',index=False)
comparison=pd.DataFrame(comparisons); comparison.to_csv(OUTPUT/'encoding_membership_comparison.csv',index=False)
show('Changes in balanced reference memberships relative to 0–1–2',comparison.loc[comparison.balance.eq('balanced')])
show('Country-sampling stability across encodings and balances',summary_all.loc[summary_all.experiment.eq('countries'),['encoding','balance','k','median_pairwise','p10_pairwise','median_reference']])
display(Markdown('**Interpretation:** inspect stable patterns and alternative solutions together. Check whether the extra clusters are distinct in domain composition, primarily differ in total exposure, or isolate a single answer. Evidence that a representation works better is exploratory; the held-out countries have not been consulted.'))


## Conclusions from the completed run

All 81 reference solutions and 8,100 repetition fits completed. Of 81 candidates, 46 have pairwise median ARI above 0.8 in both perturbations. This is a screening criterion, not proof of natural populations. ARI is unweighted; cluster shares below use equal-country weights.

| Candidate | Country median / lower decile ARI | Respondent median / lower decile ARI | Largest cluster |
|---|---:|---:|---:|
| 0–1–2, balanced, k=4 | 0.930 / 0.856 | 0.960 / 0.921 | 33.8% |
| 0–1–3, worry-heavy, k=6 | 0.934 / 0.839 | 0.933 / 0.838 | 36.2% |
| 0–1–3, worry-heavy, k=7 | 0.944 / 0.872 | 0.948 / 0.884 | 36.5% |
| 0–1–3, worry-heavy, k=10 | 0.897 / 0.850 | 0.902 / 0.853 | 34.6% |
| 0–1–5, worry-heavy, k=6 | 0.876 / 0.821 | 0.891 / 0.834 | 37.2% |

**A promising richer candidate is 0–1–3, worry-heavy, k=7.** Its profiles provisionally comprise broadly low worry (36.5%), crime emphasis (14.8%), mental-health emphasis (12.4%), broadly high worry (10.4%), drinking-water/material-security emphasis (10.0%), severe-weather emphasis (8.7%) and traffic emphasis (7.2%). These labels summarize the observed category distributions and do not identify causal explanations. Harm varies across the groups but is not the main defining contrast of every profile.

**The large group is now more interpretable, although not internally uniform.** In this k=7 solution its original-scale overall worry has mean 0.39, median 0.33 and p10–p90 of 0–0.83. The broadly high-worry group has mean 1.74 and p10–p90 of 1.17–2. Thus overall worry distinguishes broad low and high intensity, addressing an ambiguity introduced by centring alone. The largest group still has individual variation; an average profile does not establish homogeneity.

**More clusters are not automatically better.** k=10 retains good stability but only reduces the largest share to 34.6%; semantic gains must justify three additional groups. Balanced 0–1–2 k=4 is a simpler comparison. 0–1–5 remains a sensitivity scale, not an established psychological distance; its worry-heavy k=6 is reproducible but has a larger largest group and weaker stability than the corresponding 0–1–3 solution.

**Limitations:** complete cases exclude 6,660 respondents (5.73% of the previous eligible cohort, 6.05% of its analysis weight), unevenly by country. All 113 countries remain. Changes relative to v2 combine cohort selection and overall-worry representation; they cannot attribute improvements to either change separately. The extra mean uses the same multiplier as an individual centred worry feature, one-sixth of the intensity contribution in the equivalent uncentred distance. This choice is exploratory. Held-out countries remain unused. No final clustering has been frozen.
